# Link prediction eval results

Decoder types are discovered automatically as subdirectories of `lp_eval/`
(currently `bilinear`, `hadamard`, ...) — adding a new decoder's output
directory is enough to pick it up, no code changes needed. Each directory
holds files named `dataset-partitions-iteration.out`.

For every discovered decoder the same report is generated: per-metric
`dataset x partitions` summary tables (`Hits@1`, `Hits@3`, `Hits@10`, `MRR`,
`F1`), the pooled Spearman correlation between `F1` and `MRR`, and
Mann-Whitney/Cohen's d significance of `MRR` across adjacent partition
counts.

`F1` is the graph-reconstruction F1 score (`Best achieved F1 score` line)
reported for that particular run's partitioning search — it is *not*
averaged across runs, it's a single per-run value like the other metrics.

A final section compares every pair of decoder types against each other,
at the `(dataset, partitions)` level, using the same Mann-Whitney/Cohen's d
machinery.

In [6]:
from __future__ import annotations

import itertools
import os
import re
from dataclasses import dataclass
from enum import Enum

import numpy as np
import pandas as pd
from scipy import stats

LP_EVAL_DIR = os.path.join("..", "lp_eval")
SIGNIFICANCE_LEVEL = 0.05
MIN_SAMPLES_FOR_TEST = 2


class Metric(str, Enum):
    HITS_1 = "Hits@1"
    HITS_3 = "Hits@3"
    HITS_10 = "Hits@10"
    MRR = "MRR"
    F1 = "F1"


ALL_METRICS: tuple[Metric, ...] = tuple(Metric)

METRIC_LINE_PATTERNS: dict[Metric, re.Pattern[str]] = {
    Metric.HITS_1: re.compile(r"Full Model - Hits@1:\s*([\d.]+)"),
    Metric.HITS_3: re.compile(r"Full Model - Hits@3:\s*([\d.]+)"),
    Metric.HITS_10: re.compile(r"Full Model - Hits@10:\s*([\d.]+)"),
    Metric.MRR: re.compile(r"Full Model - MRR:\s*([\d.]+)"),
    Metric.F1: re.compile(r"Best achieved F1 score:\s*([\d.]+)"),
}

# dataset names can themselves contain dashes (e.g. AS-Oregon, Cit-HepPh),
# so only the trailing `-partitions-iteration` is peeled off.
RUN_FILENAME_PATTERN = re.compile(
    r"^(?P<dataset>.+)-(?P<partitions>\d+)-(?P<iteration>\d+)\.out$"
)


def parse_metrics_from_log(log_text: str) -> dict[Metric, float | None]:
    metric_values: dict[Metric, float | None] = {}
    for metric, pattern in METRIC_LINE_PATTERNS.items():
        match = pattern.search(log_text)
        metric_values[metric] = (
            float(match.group(1)) if match else None
        )
    return metric_values

In [7]:
def discover_decoder_names(lp_eval_dir: str) -> list[str]:
    decoder_dirs = [
        entry.name for entry in os.scandir(lp_eval_dir) if entry.is_dir()
    ]
    return sorted(decoder_dirs)


def build_decoder_dataframe(directory: str, decoder: str) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for filename in sorted(os.listdir(directory)):
        match = RUN_FILENAME_PATTERN.match(filename)
        if match is None:
            continue
        else:
            file_path = os.path.join(directory, filename)
            with open(file_path, "r") as log_file:
                metric_values = parse_metrics_from_log(log_file.read())
            row = {
                "decoder": decoder,
                "dataset": match.group("dataset"),
                "partitions": int(match.group("partitions")),
                "iteration": int(match.group("iteration")),
            }
            row.update({m.value: v for m, v in metric_values.items()})
            rows.append(row)
    columns = ["decoder", "dataset", "partitions", "iteration"]
    columns += [metric.value for metric in ALL_METRICS]
    return pd.DataFrame(rows, columns=columns)


def load_all_decoder_results(lp_eval_dir: str) -> pd.DataFrame:
    decoder_frames = [
        build_decoder_dataframe(os.path.join(lp_eval_dir, decoder), decoder)
        for decoder in discover_decoder_names(lp_eval_dir)
    ]
    return pd.concat(decoder_frames, ignore_index=True)

In [8]:
all_results_df = load_all_decoder_results(LP_EVAL_DIR)
all_results_df

,decoder,dataset,partitions,iteration,Hits@1,Hits@3,Hits@10,MRR,F1
0,bilinear,AS-Oregon,1,0,0.1560,0.2900,0.4470,0.2552,0.333738
1,bilinear,AS-Oregon,1,1,0.1380,0.2860,0.4400,0.2390,0.337377
2,bilinear,AS-Oregon,1,2,0.1540,0.2750,0.4460,0.2449,0.340755
3,bilinear,AS-Oregon,1,3,0.1490,0.2930,0.4640,0.2503,0.338278
4,bilinear,AS-Oregon,1,4,0.1510,0.2800,0.4490,0.2478,0.341372
...,...,...,...,...,...,...,...,...,...
176,hadamard,CITESEER,1,6,0.1501,0.2627,0.4128,0.2370,0.374256
177,hadamard,CITESEER,1,7,0.1788,0.2870,0.3996,0.2588,0.359751
178,hadamard,CITESEER,1,8,0.1766,0.2936,0.4260,0.2638,0.372043
179,hadamard,CITESEER,1,9,0.1457,0.2494,0.3819,0.2229,0.342246


In [9]:
def format_mean_and_stddev(mean: float, stddev: float) -> str:
    return f"{mean * 100:.2f}% \u00b1 {stddev * 100:.2f}%"


def build_metric_summary_table(
    df: pd.DataFrame, metric: Metric
) -> pd.DataFrame:
    grouped = df.groupby(["dataset", "partitions"])[metric.value]
    stats_df = grouped.agg(["mean", "std"])
    formatted = stats_df.apply(
        lambda row: format_mean_and_stddev(row["mean"], row["std"]),
        axis=1,
    )
    table = formatted.unstack("partitions")
    table.columns = [f"P={p}" for p in table.columns]
    return table

In [10]:
@dataclass
class GroupComparison:
    label_a: str
    label_b: str
    sample_size_a: int
    sample_size_b: int
    u_statistic: float
    p_value: float
    cohens_d: float

    @property
    def is_significant(self) -> bool:
        return self.p_value < SIGNIFICANCE_LEVEL


def compute_cohens_d(values_a: np.ndarray, values_b: np.ndarray) -> float:
    pooled_variance = (
        values_a.std(ddof=1) ** 2 + values_b.std(ddof=1) ** 2
    ) / 2
    pooled_stddev = np.sqrt(pooled_variance)
    if pooled_stddev == 0:
        return float("nan")
    else:
        return (values_b.mean() - values_a.mean()) / pooled_stddev


def compare_two_groups(
    label_a: str,
    values_a: np.ndarray,
    label_b: str,
    values_b: np.ndarray,
) -> GroupComparison | None:
    has_enough_samples = (
        len(values_a) >= MIN_SAMPLES_FOR_TEST
        and len(values_b) >= MIN_SAMPLES_FOR_TEST
    )
    if not has_enough_samples:
        return None
    else:
        u_statistic, p_value = stats.mannwhitneyu(
            values_a, values_b, alternative="two-sided"
        )
        return GroupComparison(
            label_a=label_a,
            label_b=label_b,
            sample_size_a=len(values_a),
            sample_size_b=len(values_b),
            u_statistic=u_statistic,
            p_value=p_value,
            cohens_d=compute_cohens_d(values_a, values_b),
        )

In [11]:
@dataclass(frozen=True)
class RunGroup:
    dataset: str
    partitions: int


def get_metric_values(
    df: pd.DataFrame, metric: Metric, decoder: str, group: RunGroup
) -> np.ndarray:
    mask = (
        (df["decoder"] == decoder)
        & (df["dataset"] == group.dataset)
        & (df["partitions"] == group.partitions)
    )
    return df.loc[mask, metric.value].to_numpy()


def get_partition_levels(df: pd.DataFrame) -> list[int]:
    return sorted(df["partitions"].unique())


def get_adjacent_pairs(levels: list[int]) -> list[tuple[int, int]]:
    return list(zip(levels, levels[1:]))


def compare_adjacent_partitions_for_dataset(
    df: pd.DataFrame,
    decoder: str,
    metric: Metric,
    dataset: str,
    adjacent_pairs: list[tuple[int, int]],
) -> list[GroupComparison]:
    comparisons = []
    for lower_partitions, higher_partitions in adjacent_pairs:
        values_a = get_metric_values(
            df, metric, decoder, RunGroup(dataset, lower_partitions)
        )
        values_b = get_metric_values(
            df, metric, decoder, RunGroup(dataset, higher_partitions)
        )
        comparison = compare_two_groups(
            f"P={lower_partitions}",
            values_a,
            f"P={higher_partitions}",
            values_b,
        )
        if comparison is None:
            continue
        else:
            comparisons.append(comparison)
    return comparisons


def group_comparison_to_row(
    dataset: str, comparison: GroupComparison
) -> dict[str, object]:
    return {
        "dataset": dataset,
        "comparison": f"{comparison.label_a} vs {comparison.label_b}",
        "n1": comparison.sample_size_a,
        "n2": comparison.sample_size_b,
        "U": comparison.u_statistic,
        "p-value": comparison.p_value,
        "Cohen's d": comparison.cohens_d,
        "significant": comparison.is_significant,
    }


def build_adjacent_partition_comparisons(
    df: pd.DataFrame, decoder: str, metric: Metric
) -> pd.DataFrame:
    adjacent_pairs = get_adjacent_pairs(get_partition_levels(df))
    datasets = sorted(
        df.loc[df["decoder"] == decoder, "dataset"].unique()
    )
    rows = [
        group_comparison_to_row(dataset, comparison)
        for dataset in datasets
        for comparison in compare_adjacent_partitions_for_dataset(
            df, decoder, metric, dataset, adjacent_pairs
        )
    ]
    return pd.DataFrame(rows)

In [12]:
def compute_pooled_spearman(
    df: pd.DataFrame, decoder: str, metric_a: Metric, metric_b: Metric
) -> tuple[float, float, int]:
    columns = [metric_a.value, metric_b.value]
    subset = df.loc[df["decoder"] == decoder, columns].dropna()
    correlation, p_value = stats.spearmanr(
        subset[metric_a.value], subset[metric_b.value]
    )
    return correlation, p_value, len(subset)


def print_decoder_header(decoder: str, run_count: int) -> None:
    print(f"## Decoder: {decoder} ({run_count} runs)")


def generate_decoder_report(df: pd.DataFrame, decoder: str) -> None:
    subset = df.loc[df["decoder"] == decoder]
    print_decoder_header(decoder, len(subset))
    for metric in ALL_METRICS:
        print(f"Metric: {metric.value}")
        display(build_metric_summary_table(subset, metric))
    correlation, p_value, sample_size = compute_pooled_spearman(
        df, decoder, Metric.F1, Metric.MRR
    )
    print(
        f"Spearman(F1, MRR): rho={correlation:.4f}, "
        f"p={p_value:.4g}, n={sample_size}"
    )
    print("MRR significance across adjacent partition counts:")
    display(
        build_adjacent_partition_comparisons(df, decoder, Metric.MRR)
    )

In [13]:
for decoder_name in discover_decoder_names(LP_EVAL_DIR):
    generate_decoder_report(all_results_df, decoder_name)

## Decoder: bilinear (160 runs)
Metric: Hits@1


,P=1,P=2,P=4,P=8
dataset,,,,
AS-Oregon,14.99% ± 1.07%,9.70% ± 1.23%,7.83% ± 1.13%,7.57% ± 1.40%
AstroPh,56.79% ± 1.58%,38.54% ± 1.70%,32.57% ± 2.29%,27.68% ± 2.34%
CITESEER,17.70% ± 1.14%,11.08% ± 1.79%,8.08% ± 0.99%,5.76% ± 1.10%
Cit-HepPh,48.38% ± 1.47%,31.05% ± 2.81%,27.10% ± 2.54%,24.90% ± 1.91%


Metric: Hits@3


,P=1,P=2,P=4,P=8
dataset,,,,
AS-Oregon,29.22% ± 1.24%,21.21% ± 1.34%,17.64% ± 1.63%,16.60% ± 2.03%
AstroPh,77.48% ± 1.50%,58.30% ± 2.02%,51.70% ± 2.66%,44.71% ± 3.24%
CITESEER,32.25% ± 2.93%,19.69% ± 1.80%,15.54% ± 1.02%,12.01% ± 1.69%
Cit-HepPh,73.21% ± 1.82%,51.77% ± 4.09%,45.33% ± 4.16%,42.48% ± 2.77%


Metric: Hits@10


...

,P=1
dataset,
AstroPh,32.34% ± 4.13%
CITESEER,22.92% ± 2.80%
Cit-HepPh,nan% ± nan%


Metric: F1


,P=1
dataset,
AstroPh,70.59% ± 0.05%
CITESEER,36.20% ± 1.07%
Cit-HepPh,nan% ± nan%


Spearman(F1, MRR): rho=0.7068, p=0.000494, n=20
MRR significance across adjacent partition counts:


""


# Decoder comparison

Every pair of decoder types is compared against each other metric by
metric, restricted to the `(dataset, partitions)` groups both decoders
have data for (so a decoder whose sweep hasn't finished yet only
contributes the groups it actually has runs for).

In [14]:
def get_common_groups(
    df: pd.DataFrame, decoder_a: str, decoder_b: str
) -> list[RunGroup]:
    key_columns = ["dataset", "partitions"]
    groups_a = df.loc[df["decoder"] == decoder_a, key_columns]
    groups_b = df.loc[df["decoder"] == decoder_b, key_columns]
    tuples_a = set(map(tuple, groups_a.values))
    tuples_b = set(map(tuple, groups_b.values))
    common_tuples = sorted(tuples_a & tuples_b)
    return [
        RunGroup(dataset=dataset, partitions=int(partitions))
        for dataset, partitions in common_tuples
    ]


def compare_decoders_for_group(
    df: pd.DataFrame,
    decoder_a: str,
    decoder_b: str,
    metric: Metric,
    group: RunGroup,
) -> GroupComparison | None:
    values_a = get_metric_values(df, metric, decoder_a, group)
    values_b = get_metric_values(df, metric, decoder_b, group)
    return compare_two_groups(decoder_a, values_a, decoder_b, values_b)


def decoder_comparison_to_row(
    group: RunGroup, comparison: GroupComparison
) -> dict[str, object]:
    return {
        "dataset": group.dataset,
        "partitions": group.partitions,
        "comparison": f"{comparison.label_a} vs {comparison.label_b}",
        "n1": comparison.sample_size_a,
        "n2": comparison.sample_size_b,
        "U": comparison.u_statistic,
        "p-value": comparison.p_value,
        "Cohen's d": comparison.cohens_d,
        "significant": comparison.is_significant,
    }


def build_decoder_comparison_table(
    df: pd.DataFrame, decoder_a: str, decoder_b: str, metric: Metric
) -> pd.DataFrame:
    common_groups = get_common_groups(df, decoder_a, decoder_b)
    comparisons = (
        (
            group,
            compare_decoders_for_group(
                df, decoder_a, decoder_b, metric, group
            ),
        )
        for group in common_groups
    )
    rows = [
        decoder_comparison_to_row(group, comparison)
        for group, comparison in comparisons
        if comparison is not None
    ]
    return pd.DataFrame(rows)

In [15]:
def print_decoder_pair_header(decoder_a: str, decoder_b: str) -> None:
    print(f"## {decoder_a} vs {decoder_b}")


def generate_decoder_comparison_report(
    df: pd.DataFrame, decoder_a: str, decoder_b: str
) -> None:
    print_decoder_pair_header(decoder_a, decoder_b)
    for metric in ALL_METRICS:
        table = build_decoder_comparison_table(
            df, decoder_a, decoder_b, metric
        )
        has_common_groups = not table.empty
        if not has_common_groups:
            print(f"{metric.value}: no overlapping groups yet")
            continue
        else:
            print(f"Metric: {metric.value}")
            display(table)

In [16]:
decoder_names = discover_decoder_names(LP_EVAL_DIR)
for decoder_a, decoder_b in itertools.combinations(decoder_names, 2):
    generate_decoder_comparison_report(all_results_df, decoder_a, decoder_b)

## bilinear vs hadamard
Metric: Hits@1


,dataset,partitions,comparison,n1,n2,U,p-value,Cohen's d,significant
0,AstroPh,1,bilinear vs hadamard,10,10,100.0,0.000182,-13.407253,True
1,CITESEER,1,bilinear vs hadamard,10,10,88.0,0.004420,-1.634641,True


Metric: Hits@3


,dataset,partitions,comparison,n1,n2,U,p-value,Cohen's d,significant
0,AstroPh,1,bilinear vs hadamard,10,10,100.0,0.000182,-11.116156,True
1,CITESEER,1,bilinear vs hadamard,10,10,94.0,0.000999,-1.931258,True


Metric: Hits@10


,dataset,partitions,comparison,n1,n2,U,p-value,Cohen's d,significant
0,AstroPh,1,bilinear vs hadamard,10,10,100.0,0.000182,-8.471094,True
1,CITESEER,1,bilinear vs hadamard,10,10,97.5,0.000375,-2.402573,True


Metric: MRR


,dataset,partitions,comparison,n1,n2,U,p-value,Cohen's d,significant
0,AstroPh,1,bilinear vs hadamard,10,10,100.0,0.000183,-11.874912,True
1,CITESEER,1,bilinear vs hadamard,10,10,97.0,0.000440,-1.987049,True


Metric: F1


,dataset,partitions,comparison,n1,n2,U,p-value,Cohen's d,significant
0,AstroPh,1,bilinear vs hadamard,10,10,70.0,0.140465,-0.408099,False
1,CITESEER,1,bilinear vs hadamard,10,10,46.0,0.791337,0.250220,False
